# Notebook 09 — Cross-Hardware Policy Portability

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 08 simulated streaming runtime adaptation.

Notebook 09 asks whether the same adaptive selector transfers across hardware profiles:

- x86 scalar baseline
- x86 AVX2
- x86 AVX512
- ARM64 NEON
- cloud VM baseline

Constraint view:
> a policy is portable only if structure survives architecture changes.

## Goals

1. Load Notebook 08 streaming/runtime outputs when available.
2. Define hardware profiles and architecture-specific modifiers.
3. Simulate throughput and hardware pressure across profiles.
4. Compare selector stability across hardware.
5. Measure policy portability:
   - selector agreement rate
   - throughput rank stability
   - pressure-shift sensitivity
   - architecture-specific regime changes
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 08 streaming output

If missing, create a compact fallback window table.

In [ ]:
stream_path = RESULTS_DIR / "notebook08_streaming_runtime_adaptation.csv"

if stream_path.exists():
    stream = pd.read_csv(stream_path)
    print("Loaded:", stream_path)
else:
    print("Notebook 08 output not found; using fallback streaming table.")
    regimes = [
        "low_entropy_repeating", "sequential_ids", "uniform_32bit",
        "zipfian_smallints", "clustered_ranges", "sequential_ids",
        "uniform_32bit", "low_entropy_repeating"
    ]
    rows = []
    for i, reg in enumerate(regimes):
        for j in range(4):
            rows.append({
                "window_id": len(rows),
                "truth_regime": reg,
                "adaptive_policy": {
                    "low_entropy_repeating": "coherent_local",
                    "sequential_ids": "hybrid",
                    "uniform_32bit": "simd",
                    "zipfian_smallints": "hybrid",
                    "clustered_ranges": "guarded_fallback",
                }[reg],
                "coherence_score": {
                    "low_entropy_repeating": 0.98,
                    "sequential_ids": 0.56,
                    "uniform_32bit": 0.04,
                    "zipfian_smallints": 0.35,
                    "clustered_ranges": 0.12,
                }[reg],
                "hardware_pressure_proxy": {
                    "low_entropy_repeating": 0.01,
                    "sequential_ids": 0.25,
                    "uniform_32bit": 0.91,
                    "zipfian_smallints": 0.78,
                    "clustered_ranges": 0.99,
                }[reg],
            })
    stream = pd.DataFrame(rows)

stream.head()

## Define hardware profiles

These are portable simulation profiles. Later versions can replace modifiers with measured benchmark/counter outputs.

In [ ]:
hardware_profiles = pd.DataFrame([
    {
        "hardware_profile": "scalar_reference",
        "architecture": "x86_64",
        "simd_width": 1,
        "scalar_bonus": 1.10,
        "simd_bonus": 0.70,
        "branch_penalty": 1.00,
        "cache_penalty": 1.00,
        "switch_penalty": 0.03,
    },
    {
        "hardware_profile": "avx2_linux",
        "architecture": "x86_64",
        "simd_width": 8,
        "scalar_bonus": 1.00,
        "simd_bonus": 1.25,
        "branch_penalty": 1.05,
        "cache_penalty": 1.00,
        "switch_penalty": 0.035,
    },
    {
        "hardware_profile": "avx512_linux",
        "architecture": "x86_64",
        "simd_width": 16,
        "scalar_bonus": 0.95,
        "simd_bonus": 1.45,
        "branch_penalty": 1.12,
        "cache_penalty": 1.06,
        "switch_penalty": 0.045,
    },
    {
        "hardware_profile": "neon_arm64",
        "architecture": "arm64",
        "simd_width": 4,
        "scalar_bonus": 1.05,
        "simd_bonus": 1.10,
        "branch_penalty": 0.95,
        "cache_penalty": 0.92,
        "switch_penalty": 0.03,
    },
    {
        "hardware_profile": "cloud_vm_baseline",
        "architecture": "virtualized",
        "simd_width": 4,
        "scalar_bonus": 0.90,
        "simd_bonus": 0.95,
        "branch_penalty": 1.20,
        "cache_penalty": 1.25,
        "switch_penalty": 0.06,
    },
])

hardware_profiles

## Hardware-aware policy simulation

The model re-scores candidate policies per hardware profile. It is intentionally transparent:

- SIMD gains increase with SIMD profile strength.
- Branch/cache penalties reduce benefit under fragmented regimes.
- Cloud VM baseline penalizes pressure and switching.
- ARM64 NEON slightly favors lower cache/branch pressure.

In [ ]:
base_policy_table = {
    "low_entropy_repeating": {"scalar": 1650, "simd": 1400, "coherent_local": 1750, "guarded_fallback": 1200, "hybrid": 1500},
    "sequential_ids": {"scalar": 1350, "simd": 1450, "coherent_local": 1300, "guarded_fallback": 1100, "hybrid": 1400},
    "uniform_32bit": {"scalar": 1100, "simd": 1900, "coherent_local": 1000, "guarded_fallback": 1200, "hybrid": 1650},
    "zipfian_smallints": {"scalar": 1200, "simd": 1500, "coherent_local": 1250, "guarded_fallback": 1150, "hybrid": 1550},
    "clustered_ranges": {"scalar": 900, "simd": 950, "coherent_local": 850, "guarded_fallback": 1200, "hybrid": 1050},
}

policies = ["scalar", "simd", "coherent_local", "guarded_fallback", "hybrid"]

def adjusted_throughput(regime, policy, hw, pressure):
    base = base_policy_table.get(regime, {}).get(policy, 1000)

    if policy == "scalar":
        mult = hw["scalar_bonus"]
    elif policy == "simd":
        mult = hw["simd_bonus"]
    elif policy == "coherent_local":
        mult = 0.5 * hw["scalar_bonus"] + 0.5 * (1.0 / max(hw["cache_penalty"], 0.1))
    elif policy == "guarded_fallback":
        mult = 1.0 / (0.75 * hw["branch_penalty"] + 0.25 * hw["cache_penalty"])
    else:
        mult = 0.45 * hw["scalar_bonus"] + 0.45 * hw["simd_bonus"] + 0.10

    pressure_penalty = 1.0 - 0.18 * pressure * (0.5 * hw["branch_penalty"] + 0.5 * hw["cache_penalty"] - 1.0)
    return max(1.0, base * mult * pressure_penalty)

rows = []
for _, hw in hardware_profiles.iterrows():
    prev_policy = None
    for _, row in stream.iterrows():
        regime = row["truth_regime"]
        pressure = float(row.get("hardware_pressure_proxy", 0.5))
        coherence = float(row.get("coherence_score", 0.5))

        scores = {}
        throughputs = {}
        for pol in policies:
            tp = adjusted_throughput(regime, pol, hw, pressure)
            throughputs[pol] = tp

            # Score mixes throughput with coherence/pressure suitability.
            if pol == "coherent_local":
                score = tp * (1.0 + 0.20 * coherence - 0.10 * pressure)
            elif pol == "guarded_fallback":
                score = tp * (1.0 + 0.25 * pressure)
            elif pol == "simd":
                score = tp * (1.0 + 0.10 * hw["simd_width"] / 16.0 - 0.08 * pressure)
            elif pol == "scalar":
                score = tp * (1.0 + 0.10 * coherence)
            else:
                score = tp * (1.0 + 0.05 * (1.0 - abs(coherence - pressure)))

            if prev_policy is not None and pol != prev_policy:
                score *= (1.0 - hw["switch_penalty"])
            scores[pol] = score

        selected_policy = max(scores, key=scores.get)
        selected_tp = throughputs[selected_policy]
        if prev_policy is not None and selected_policy != prev_policy:
            selected_tp *= (1.0 - hw["switch_penalty"])
        prev_policy = selected_policy

        rows.append({
            "hardware_profile": hw["hardware_profile"],
            "architecture": hw["architecture"],
            "window_id": row["window_id"],
            "truth_regime": regime,
            "coherence_score": coherence,
            "hardware_pressure_proxy": pressure,
            "selected_policy": selected_policy,
            "selected_throughput": selected_tp,
            "best_possible_policy": max(throughputs, key=throughputs.get),
            "best_possible_throughput": max(throughputs.values()),
            "fixed_scalar_throughput": throughputs["scalar"],
            "fixed_simd_throughput": throughputs["simd"],
            "selected_efficiency": selected_tp / max(throughputs.values()),
        })

portable = pd.DataFrame(rows)
portable.head()

## Compute portability metrics

In [ ]:
# Reference policy from scalar_reference per window.
ref = (
    portable[portable["hardware_profile"] == "scalar_reference"]
    [["window_id", "selected_policy"]]
    .rename(columns={"selected_policy": "reference_policy"})
)

portable = portable.merge(ref, on="window_id", how="left")
portable["matches_reference_policy"] = portable["selected_policy"] == portable["reference_policy"]

summary = (
    portable
    .groupby("hardware_profile", as_index=False)
    .agg(
        architecture=("architecture", "first"),
        mean_throughput=("selected_throughput", "mean"),
        mean_efficiency=("selected_efficiency", "mean"),
        policy_match_reference_rate=("matches_reference_policy", "mean"),
        mean_pressure=("hardware_pressure_proxy", "mean"),
        policy_switches=("selected_policy", lambda x: int(pd.Series(x).ne(pd.Series(x).shift(1)).sum())),
    )
)

# Throughput rank by hardware profile.
summary["throughput_rank"] = summary["mean_throughput"].rank(ascending=False, method="dense").astype(int)
summary

## Export portability tables

In [ ]:
csv_path = RESULTS_DIR / "notebook09_cross_hardware_policy_portability.csv"
json_path = RESULTS_DIR / "notebook09_cross_hardware_policy_portability.json"
summary_csv_path = RESULTS_DIR / "notebook09_cross_hardware_policy_summary.csv"

portable.to_csv(csv_path, index=False)
portable.to_json(json_path, orient="records", indent=2)
summary.to_csv(summary_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", summary_csv_path)

## Figure 1 — Mean throughput by hardware profile

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook09_mean_throughput_by_hardware.png"

plot_df = summary.sort_values("mean_throughput")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["hardware_profile"], plot_df["mean_throughput"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean selected throughput")
plt.title("Cross-Hardware Portability: Mean Throughput")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Policy portability vs reference

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook09_policy_match_reference.png"

plot_df = summary.sort_values("policy_match_reference_rate")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["hardware_profile"], plot_df["policy_match_reference_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Match rate vs scalar_reference policy")
plt.title("Cross-Hardware Policy Portability")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Policy timeline by hardware profile

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook09_policy_timeline_by_hardware.png"

policy_labels = sorted(portable["selected_policy"].unique())
policy_to_id = {p: i for i, p in enumerate(policy_labels)}

plt.figure(figsize=(12, 6))
for hw in portable["hardware_profile"].unique():
    part = portable[portable["hardware_profile"] == hw]
    y = part["selected_policy"].map(policy_to_id)
    plt.step(part["window_id"], y + 0.03 * list(portable["hardware_profile"].unique()).index(hw), where="mid", label=hw)

plt.yticks(list(policy_to_id.values()), list(policy_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Selected policy")
plt.title("Selected Policy Timeline by Hardware")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Efficiency by regime and hardware

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook09_efficiency_by_regime_hardware.png"

pivot = portable.pivot_table(
    index="truth_regime",
    columns="hardware_profile",
    values="selected_efficiency",
    aggfunc="mean"
)

plt.figure(figsize=(10, 5))
plt.imshow(pivot.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
plt.colorbar(label="Mean selected / best possible throughput")
plt.title("Selector Efficiency by Regime and Hardware")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Portability summary matrix

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook09_portability_summary_matrix.png"

mat_cols = [
    "mean_throughput",
    "mean_efficiency",
    "policy_match_reference_rate",
    "policy_switches",
]

mat = summary.set_index("hardware_profile")[mat_cols].copy()
for col in mat_cols:
    lo, hi = mat[col].min(), mat[col].max()
    if hi != lo:
        mat[col] = (mat[col] - lo) / (hi - lo)
    else:
        mat[col] = 0.0

plt.figure(figsize=(8, 5))
plt.imshow(mat.values, aspect="auto")
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(mat.columns)), mat.columns, rotation=45, ha="right")
plt.colorbar(label="Normalized score")
plt.title("Cross-Hardware Portability Summary Matrix")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_09_cross_hardware_policy_portability.md"

policy_counts = pd.crosstab(portable["hardware_profile"], portable["selected_policy"])

lines = [
    "# Report 09 — Cross-Hardware Policy Portability",
    "",
    "This report tests whether adaptive execution policies remain stable across simulated hardware profiles.",
    "",
    "Constraint view:",
    "> a policy is portable only if structure survives architecture changes.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Summary CSV: `{summary_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Hardware portability summary",
    "",
    summary.to_markdown(index=False),
    "",
    "## Policy counts by hardware",
    "",
    policy_counts.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Policy portability measures whether the same structural signal produces stable decisions across architectures.",
    "- AVX-style profiles should favor SIMD more often, while scalar and virtualized profiles penalize pressure differently.",
    "- Low portability is not failure; it identifies architecture-specific constraints.",
    "- This notebook creates a bridge from adaptive selection to architecture-aware scheduling.",
    "",
    "## Next step",
    "",
    "Notebook 10 can build a learned execution selector from the policy table and evaluate whether learned routing improves over transparent rules.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook09_cross_hardware_policy_portability_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook09_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_09_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))